# Analyse: "In welcher Blase bin ich denn hier gelandet? (2026)"

## Beeinflussen Bundesländer die Umfrageantworten?

Dieses Notebook ergänzt das Kurs-Notebook *In welcher Blase bin ich denn hier gelandet - 2026* um zwei zusätzliche Fragen:

1. Lässt sich im Kursdatensatz ein Bundesland-Effekt auf Umfrageantworten zeigen?
2. Wie hängen politische Antwortmuster auf Bundesland-Ebene mit dem Ausmaß der ländlichen Besiedlung zusammen?

Dafür wird zusätzlich eine Datenbank mit dem Anteil der ländlichen Bevölkerung je Bundesland aufgebaut (`data/bundeslaender_rural.csv`) und eine separate Korrelationsanalyse durchgeführt.

## 1. Inhaltliche Analyse des Ausgangs-Notebooks

Das Notebook ist als didaktische Einführung in empirisches Arbeiten mit R/tidyverse angelegt. Inhaltlich passiert Folgendes:

**Datengrundlage.** Geladen wird `survey2026-04_colab.rds` (101 Befragte, 24 Variablen). Die Daten stammen aus einer Kursumfrage und sind ausdrücklich anonymisiert.

**Variablen.** Der Datensatz enthält:

- Soziodemografie: `geschlecht` (Männlich / Weiblich / Andere)
- Parteipräferenzen: `vote` (Sonntagsfrage) sowie Skalometer `skalo_cdu`, `skalo_spd`, `skalo_pds`, `skalo_gru`, `skalo_fdp`, `skalo_afd`, `skalo_bsw` (je -5 … +5)
- Politische Selbstverortung: `lire_self` (links-rechts, 1–11), `econ_self` (Sozialstaat, 1–11), `immi_self` (Migration, 1–11), `klim_self` (Klima, 1–11), `euin_self` (EU, 1–11)
- Demokratiebewertungen: `dem_brd`, `dem_usa`, `dem_hun`, `dem_pol`, `dem_rus`
- Design-Items zu Regierungsformen: `design_introddem`, `design_intromw`, `design_minreg`, `design_direktbk`, `design_manyparty`

**Analyseschritte.** Das Notebook

1. führt in `count()`, Pipes und `mutate()` ein,
2. rechnet Anteile für die Sonntagsfrage aus und visualisiert sie mit `ggplot2` und eigenen Parteifarben,
3. vergleicht die Links-Rechts-Selbsteinstufung zwischen Frauen und Männern mittels Dichteplot,
4. zeigt mit einem Streudiagramm (`econ_self` × `immi_self`, farbig nach Wahlabsicht) zwei Konfliktdimensionen gleichzeitig,
5. fordert am Ende zum eigenen Explorieren auf.

**Pädagogische Kernbotschaften.** Das Notebook betont drei Punkte, die für die Bundesland-Frage direkt relevant sind:

- *Datenkontext*: Die Grafik zeigt den Kurs, nicht „die Studierenden in Deutschland“ und nicht „die Jugend“.
- *Datenschutz*: Zeitstempel, **Geburtsbundesland** und offene Freitexte wurden bewusst entfernt. Seltene Parteien sind zusammengefasst.
- *Hypothesenarbeit*: Beschreibung ist Ausgangspunkt, Hypothesenprüfung ist das Ziel.

## 2. Setup

In [ ]:
library(tidyverse)

In [ ]:
# Kursumfrage laden (wie im Ausgangs-Notebook)
if (!file.exists("survey2026-04_colab.rds")) {
  download.file(
    "https://raw.githubusercontent.com/cstecker/politicsRLab/main/data/survey2026-04_colab.rds",
    "survey2026-04_colab.rds",
    mode = "wb"
  )
}
survey <- readRDS("survey2026-04_colab.rds")
glimpse(survey)

## 3. Beeinflussen Bundesländer die Antworten im Kursdatensatz?

Die direkte, ehrliche Antwort steht schon im Ausgangs-Notebook: **Die anonymisierte Datei enthält kein Bundesland.** Sie können das nachprüfen:

In [ ]:
# Suchen nach bundeslandähnlichen Spaltennamen
bundesland_vars <- names(survey) |>
  stringr::str_subset("(?i)bundesland|land|wohn|ort|region|rural|urban|plz")
bundesland_vars

names(survey)

Das Ergebnis ist leer: In `survey` gibt es weder `bundesland`, noch Wohnort, PLZ oder Region. Damit kann aus diesem Datensatz **keine gültige Aussage** darüber getroffen werden, ob das Bundesland die Antworten beeinflusst. Das ist keine Datenlücke aus Versehen, sondern Datenschutz mit Absicht.

Was bedeutet das methodisch?

- Jede Gruppierung nach Bundesland wäre hier nur geraten oder über eine Drittvariable konstruiert. Das wäre **nicht valide**.
- Statt die Frage fallen zu lassen, verlegen wir die Analyseebene: Wir schauen, **wie politische Muster zwischen Bundesländern** aussehen — auf Basis öffentlicher, bereits aggregierter Daten. Das beantwortet eine leicht andere, aber ehrlicher beantwortbare Frage.

## 4. Datenbank: Ländliche Besiedlung je Bundesland

Die Datei `data/bundeslaender_rural.csv` enthält für alle 16 Bundesländer:

- `population_2023`, `area_km2`, `density_per_km2` — harte Zählwerte (Destatis, Stand 2023).
- `rural_share_pct` — Anteil der Bevölkerung in ländlich geprägten Kreisen. Die Werte orientieren sich an der BBSR-Typisierung „Siedlungsstrukturelle Kreistypen“ (2021), gerundet; die Stadtstaaten Berlin, Bremen, Hamburg sind per Definition auf 0 gesetzt.
- `east_west` — Ost (inkl. Berlin) vs. West, nützlich, weil der Ost-West-Unterschied sich stark mit Ruralität überlagert.

Beide Metriken (Einwohnerdichte, Ruralitätsanteil) sind gleichzeitig interessant, weil sie unterschiedlich gewichten: Die Dichte bestraft Stadtstaaten sehr stark, der BBSR-Anteil differenziert dagegen innerhalb der Flächenländer.

In [ ]:
rural <- read_csv("data/bundeslaender_rural.csv", show_col_types = FALSE)
rural

In [ ]:
rural |>
  mutate(bundesland = forcats::fct_reorder(bundesland, rural_share_pct)) |>
  ggplot(aes(x = bundesland, y = rural_share_pct, fill = east_west)) +
  geom_col() +
  coord_flip() +
  scale_fill_manual(values = c("Ost" = "#2A6DF4", "West" = "#D8A21B")) +
  labs(
    title = "Ländlichkeit nach Bundesland",
    subtitle = "Anteil der Bevölkerung in ländlich geprägten Kreisen (BBSR-Typisierung)",
    x = NULL, y = "Anteil ländlich (%)", fill = NULL
  ) +
  theme_minimal(base_size = 13)

## 5. Separate Korrelationsanalyse: Ländlichkeit und politische Antworten je Bundesland

Weil der Kursdatensatz kein Bundesland enthält, nehmen wir **aggregierte Umfrage-/Wahldaten**: die Zweitstimmenanteile der Bundestagswahl 2021 je Bundesland (Quelle: Bundeswahlleiterin; `data/bundeslaender_btw2021.csv`). Diese Antworten sind methodisch einer Sonntagsfrage nahe genug, um die Grundfrage zu beantworten: *Lässt sich ein Bundesland-Effekt erkennen, und geht er mit Ländlichkeit einher?*

In [ ]:
btw <- read_csv("data/bundeslaender_btw2021.csv", show_col_types = FALSE)

bl <- rural |>
  left_join(btw, by = "bundesland_short")

bl

In [ ]:
# Pearson-Korrelation von Ruralität und Parteianteilen
parties <- c("union_pct", "spd_pct", "gruene_pct", "fdp_pct", "afd_pct", "linke_pct")

cor_table <- tibble(
  partei = parties,
  r_rural   = sapply(parties, function(p) cor(bl$rural_share_pct, bl[[p]])),
  r_density = sapply(parties, function(p) cor(bl$density_per_km2, bl[[p]]))
) |>
  mutate(across(starts_with("r_"), ~ round(.x, 2)))

cor_table

Die Korrelationen sind auf n = 16 Bundesländern berechnet — das ist sehr klein. Einzelne Länder (vor allem die Stadtstaaten und die AfD-Spitzenwerte in den ostdeutschen Flächenländern) hebeln die Koeffizienten stark. Deshalb zusätzlich ein Scatterplot mit Ost-West-Kennzeichnung:

In [ ]:
bl |>
  ggplot(aes(x = rural_share_pct, y = afd_pct, color = east_west)) +
  geom_point(size = 3) +
  ggrepel::geom_text_repel(aes(label = bundesland_short), size = 3.5, show.legend = FALSE) +
  geom_smooth(method = "lm", se = FALSE, color = "grey40", linewidth = 0.5) +
  scale_color_manual(values = c("Ost" = "#2A6DF4", "West" = "#D8A21B")) +
  labs(
    title = "Ländlichkeit und AfD-Zweitstimmenanteil (BTW 2021)",
    x = "Anteil ländlich (%)", y = "AfD-Zweitstimmen (%)", color = NULL
  ) +
  theme_minimal(base_size = 13)

In [ ]:
bl |>
  ggplot(aes(x = rural_share_pct, y = gruene_pct, color = east_west)) +
  geom_point(size = 3) +
  ggrepel::geom_text_repel(aes(label = bundesland_short), size = 3.5, show.legend = FALSE) +
  geom_smooth(method = "lm", se = FALSE, color = "grey40", linewidth = 0.5) +
  scale_color_manual(values = c("Ost" = "#2A6DF4", "West" = "#D8A21B")) +
  labs(
    title = "Ländlichkeit und Grünen-Zweitstimmenanteil (BTW 2021)",
    x = "Anteil ländlich (%)", y = "Grüne-Zweitstimmen (%)", color = NULL
  ) +
  theme_minimal(base_size = 13)

### Teilkorrelationen innerhalb West bzw. Ost

Weil die Ost-West-Achse viele Parteieffekte erklärt, lohnt sich die Korrelation getrennt nach Landesteil:

In [ ]:
by_ow <- bl |>
  group_by(east_west) |>
  summarise(
    r_afd    = cor(rural_share_pct, afd_pct),
    r_union  = cor(rural_share_pct, union_pct),
    r_gruene = cor(rural_share_pct, gruene_pct),
    r_spd    = cor(rural_share_pct, spd_pct),
    n = n()
  ) |>
  mutate(across(starts_with("r_"), ~ round(.x, 2)))

by_ow

## 6. Interpretation und Grenzen

**Was wir zeigen können.**

- Auf Bundesland-Ebene gibt es ein klares Muster: Je höher der Anteil ländlicher Besiedlung, desto höher tendenziell der AfD-Zweitstimmenanteil und desto niedriger der Grünen-Anteil. Das deckt sich mit breit dokumentierter Wahlforschung zur Stadt-Land-Kluft.
- Der Effekt ist **stark mit der Ost-West-Achse konfundiert**: Die ostdeutschen Flächenländer sind gleichzeitig sehr ländlich *und* haben AfD-Hochburgen. Die Teilkorrelationen innerhalb West und innerhalb Ost sind deutlich kleiner als die Gesamtkorrelation — klassischer Confounder.
- n = 16 ist klein. Konfidenzintervalle wären hier breit, einzelne Ausreißer (Saarland, Stadtstaaten) verschieben das Bild sichtbar.

**Was wir *nicht* zeigen können.**

- Ob Bundesländer die **Antworten in der Kursumfrage** beeinflussen, lässt sich aus `survey2026-04_colab.rds` nicht beantworten: Das Bundesland wurde zu Recht anonymisiert entfernt.
- Aggregierte Korrelationen auf Bundesland-Ebene sind **kein Ersatz** für Individualdaten. Ein ökologischer Fehlschluss wäre, aus der Landesebene auf einzelne Personen zu schließen.

**Wenn man die Frage wirklich im Kursdatensatz beantworten wollte**, bräuchte es eine nicht-anonymisierte Version mit `bundesland` als Variable, gefolgt von z. B. `group_by(bundesland) |> summarise(across(c(lire_self, immi_self, econ_self), mean, na.rm = TRUE))` und einem ANOVA-artigen Test oder einer Regression mit Bundesland als Faktor. Wegen des Datenschutzes und der kleinen Fallzahl pro Bundesland (oft < 10) wäre das Ergebnis selbst dann mit Vorsicht zu lesen.